# OrthodoxAI - Speech to Text

> Konstantinos Mpouros <br>
> Github: https://github.com/konstantinosmpouros?tab=repositories<br>
> Year: 2025

## About the Project

The **OrthodoxAI Speech to Text** is a project aimed at transcribing Orthodox sermons and speeches from MP3 audio files into accurate text format. The transcription process will be powered by the **Eleven Labs**, ensuring high accuracy and efficiency. By leveraging this advanced AI-driven speech recognition technology, the project will provide **precise and structured transcriptions**, preserving Orthodox teachings for research, study, and digital archiving.


## Libraries

In [1]:
# Data handling and manipulation
import pandas as pd
from utils import (
    search_for_audio_files,
    extract_theme,
    get_audio_file,
    sum_audio_duration
)

# Speech to Text API
from elevenlabs import ElevenLabs
from openai import OpenAI

# Transcription and logging
import os
import random
import time
import json
from pathlib import Path
import logging
from tqdm import tqdm
import httpx

from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import BoundedSemaphore
try:
    from elevenlabs.api.error import APIError
except ImportError:
    APIError = Exception  # fallback if the import path changes

# Load the API keys
from dotenv import load_dotenv, find_dotenv
dotenv_path = find_dotenv()
_ = load_dotenv(dotenv_path)

## Data Extraction

### Athanasios Mitilinaios

In [2]:
data_path = "../data/Omilies/speeches_athanasios_mitilinaios"
knowledge_base_path = "./knowledge_base/Orthodox/Omilies/speeches_athanasios_mitilinaios"

In [3]:
athanasios_mitilinaios = search_for_audio_files(data_path)
athanasios_mitilinaios["theme"] = athanasios_mitilinaios["file_name"].apply(extract_theme)
athanasios_mitilinaios.sample(10)

,file_name,file_path,theme
3089,3083_05-10-86_ΣΥΓΧΡΟΝΑ_ΚΑΥΤΑ_ΘΕΜΑΤΑ_π_ΑΘ_ΜΥΤΙΛ...,../data/Omilies/speeches_athanasios_mitilinaio...,ΣΥΓΧΡΟΝΑ_ΚΑΥΤΑ_ΘΕΜΑΤΑ
3649,3643_ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ_π_ΑΘ_...,../data/Omilies/speeches_athanasios_mitilinaio...,ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ
2992,2986_1990_ΔΙΑΚΑΙΝΗΣΙΜΟΣ_ΕΒΔΟΜΑΣ_π_ΑΘ_ΜΥΤΙΛΗΝΑΙ...,../data/Omilies/speeches_athanasios_mitilinaio...,1990_ΔΙΑΚΑΙΝΗΣΙΜΟΣ_ΕΒΔΟΜΑΣ
148,0142_12-01-81_ΔΕΥΤΕΡΟΝΟΜΙΟΝ_π_ΑΘ_ΜΥΤΙΛΗΝΑΙΟΥ.mp3,../data/Omilies/speeches_athanasios_mitilinaio...,ΔΕΥΤΕΡΟΝΟΜΙΟΝ
4299,4293_ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_π_ΑΘ_ΜΥΤΙΛΗΝΑΙΟΥ.mp3,../data/Omilies/speeches_athanasios_mitilinaio...,ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ
3203,3197_ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ_π_ΑΘ_...,../data/Omilies/speeches_athanasios_mitilinaio...,ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ
2142,2136_15-10-89_ΚΥΡΙΑΚΗ_Δ_ΛΟΥΚΑ_π_ΑΘ_ΜΥΤΙΛΗΝΑΙΟΥ...,../data/Omilies/speeches_athanasios_mitilinaio...,ΚΥΡΙΑΚΗ_Δ_ΛΟΥΚΑ
2309,2303_27-11-94_ΚΥΡΙΑΚΗ_ΙΓ_ΛΟΥΚΑ_π_ΑΘ_ΜΥΤΙΛΗΝΑΙΟ...,../data/Omilies/speeches_athanasios_mitilinaio...,ΚΥΡΙΑΚΗ_ΙΓ_ΛΟΥΚΑ
936,0930_12-02-89_ΠΡΑΞΕΙΣ_ΤΩΝ_ΑΠΟΣΤΟΛΩΝ_π_ΑΘ_ΜΥΤΙΛ...,../data/Omilies/speeches_athanasios_mitilinaio...,ΠΡΑΞΕΙΣ_ΤΩΝ_ΑΠΟΣΤΟΛΩΝ
4279,4273_ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_π_ΑΘ_ΜΥΤΙΛΗΝΑΙΟΥ.mp3,../data/Omilies/speeches_athanasios_mitilinaio...,ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ


In [4]:
athanasios_mitilinaios['theme'].nunique()

278

In [5]:
theme_counts = athanasios_mitilinaios["theme"].value_counts()
frequent_themes = theme_counts[theme_counts > 10]
pd.DataFrame(frequent_themes)

,count
theme,
ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ,1017
ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ,321
ΣΟΦΙΑ_ΣΕΙΡΑΧ,296
ΠΡΑΞΕΙΣ_ΤΩΝ_ΑΠΟΣΤΟΛΩΝ,263
ΚΑΤΗΧΗΣΕΙΣ_ΑΓΙΟΥ_ΚΥΡΙΛΛΟΥ,211
ΕΙΣ_ΤΗΝ_ΥΠΕΡΑΓΙΑΝ_ΘΕΟΤΟΚΟΝ,110
ΙΕΡΑ_ΑΠΟΚΑΛΥΨΙΣ,103
ΕΙΣ_ΠΡΟΣΚΥΝΗΤΑΣ,102
ΠΡΟΦΗΤΗΣ_ΗΣΑΙΑΣ,92


In [6]:
example = athanasios_mitilinaios.sample(5)
example

,file_name,file_path,theme
3874,3868_ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ_π_ΑΘ_...,../data/Omilies/speeches_athanasios_mitilinaio...,ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ
2019,2013_07-09-86_ΚΥΡΙΑΚΗ_ΠΡΟ_ΤΗΣ_ΥΨΩΣΕΩΣ_π_ΑΘ_ΜΥΤ...,../data/Omilies/speeches_athanasios_mitilinaio...,ΚΥΡΙΑΚΗ_ΠΡΟ_ΤΗΣ_ΥΨΩΣΕΩΣ
182,0176_26-04-82_ΔΕΥΤΕΡΟΝΟΜΙΟΝ_π_ΑΘ_ΜΥΤΙΛΗΝΑΙΟΥ.mp3,../data/Omilies/speeches_athanasios_mitilinaio...,ΔΕΥΤΕΡΟΝΟΜΙΟΝ
1445,1439_07-10-85_ΚΑΤΗΧΗΣΕΙΣ_ΑΓΙΟΥ_ΚΥΡΙΛΛΟΥ_π_ΑΘ_Μ...,../data/Omilies/speeches_athanasios_mitilinaio...,ΚΑΤΗΧΗΣΕΙΣ_ΑΓΙΟΥ_ΚΥΡΙΛΛΟΥ
3151,3145_ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ_π_ΑΘ_...,../data/Omilies/speeches_athanasios_mitilinaio...,ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ


In [7]:
example['file_path']

3874    ../data/Omilies/speeches_athanasios_mitilinaio...
2019    ../data/Omilies/speeches_athanasios_mitilinaio...
182     ../data/Omilies/speeches_athanasios_mitilinaio...
1445    ../data/Omilies/speeches_athanasios_mitilinaio...
3151    ../data/Omilies/speeches_athanasios_mitilinaio...
Name: file_path, dtype: object

In [8]:
print(type(example['file_path'].values))
print(example['file_path'].values[0])

<class 'numpy.ndarray'>
../data/Omilies/speeches_athanasios_mitilinaios/3868_ΑΠΑΝΤΗΣΕΙΣ_ΑΠΟΡΙΩΝ_ΑΝΩΤ_ΚΑΤΗΧΗΤΙΚΟΥ_π_ΑΘ_ΜΥΤΙΛΗΝΑΙΟΥ.mp3


In [9]:
audio_file = get_audio_file(example['file_path'].values[0])
audio_file

In [10]:
duration_seconds = sum_audio_duration(data_path)
print(f"Total duration: {duration_seconds:.2f} seconds")
print(f"Total duration: {duration_seconds / 60:.2f} minutes")

Total duration: 9114800.59 seconds
Total duration: 151913.34 minutes


## Speech to Text

### Sequential Calling

In [7]:
def elevenlabs_transcribe(audio_file, file_name):
    client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))
    result = client.speech_to_text.convert(
        model_id="scribe_v1",
        file=audio_file,
        tag_audio_events=False,
        request_options={"timeout": 900.0}
    )
    return result.text

In [8]:
def transcribe_one(file_name: str, file_path: str, transcribe_func, knowledge_base_path: str):
    """
    Transcribes one audio file and writes JSON to the knowledge base.
    Returns the file_name on success, None on failure (with logging).
    """
    out_path = Path(knowledge_base_path) / (Path(file_name).stem + ".json")

    try:
        with open(file_path, "rb") as audio_file:
            transcription_text = transcribe_func(audio_file, file_name=file_name)
    except Exception as e:
        size = Path(file_path).stat().st_size if Path(file_path).exists() else "n/a"
        logging.exception(f"Error transcribing {file_name}: {e} (size={size} bytes)")
        return None

    try:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump({"content": transcription_text}, f, ensure_ascii=False, indent=4)
        logging.info(f"Successfully transcribed and saved {file_name}.")
        return file_name
    except Exception as e:
        logging.exception(f"Error saving transcription for {file_name}: {e}")
        return None

In [9]:
def transcribe_all_sequential(data_path: str, knowledge_base_path: str, transcribe_fn):
    """
    Finds audio files, skips already-transcribed ones, and processes NEW files
    one-by-one with a tqdm progress bar. No parallel constructs are used.
    Assumes there is a function `search_for_audio_files(data_path)` that returns
    a DataFrame with columns: ['file_name', 'file_path'].
    """
    try:
        logging.info("\n\n\n\n------------------------Sequential Transcription Started------------------------")

        # Ensure output directory exists
        Path(knowledge_base_path).mkdir(parents=True, exist_ok=True)

        # Find audio files (expects a DataFrame with file_name, file_path)
        df_files = search_for_audio_files(data_path)

        # Mark which are already transcribed
        df_files["transcribed"] = df_files["file_name"].apply(
            lambda name: (Path(knowledge_base_path) / (Path(name).stem + ".json")).exists()
        )

        df_to_process = df_files[~df_files["transcribed"]]

        if df_to_process.empty:
            logging.info("All files already transcribed.")
            return

        logging.info("Transcribing %d new files.", len(df_to_process))

        # Sequential loop with tqdm
        for _, row in tqdm(
            df_to_process.iterrows(),
            total=len(df_to_process),
            desc="Processing audio files",
            unit="file",
        ):
            file_name = row["file_name"]
            file_path = row["file_path"]
            transcribe_one(file_name, file_path, transcribe_fn, knowledge_base_path)

        logging.info("All files processed.")
    except Exception as ex:
        logging.exception(f"Error in the transcription process: {ex}")
    finally:
        # Cleanly close the shared HTTPX client
        try:
            HTTPX_CLIENT.close()
        except Exception:
            pass

### Parallel Calling

In [11]:
def transcription(file_name, file_path, transcribe_func, knowledge_base_path):
    out_path = Path(knowledge_base_path) / (Path(file_name).stem + ".json")

    # Load the audio file
    try:
        audio_file = open(file_path, "rb")
    except Exception as e:
        logging.error(f"Error loading audio file {file_name}: {e}")
        return None

    # Transcribe the audio using the passed function
    try:
        transcription_text = transcribe_func(audio_file, file_name=file_name)
    except Exception as e:
        logging.error(f"Error transcribing {file_name}: {e}")
        logging.error(f"Files length: {len(audio_file)}")
        return None

    # Save transcription as JSON
    transcription_dict = {"content": transcription_text}
    try:
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(transcription_dict, f, ensure_ascii=False, indent=4)
        logging.info(f"Successfully transcribed and saved {file_name}.")
        return file_name  # Return file_name to track processed files
    except Exception as e:
        logging.error(f"Error saving transcription for {file_name}: {e}")
        return None

In [12]:
def parallel_transcription(bata_path, knowledge_base_path, transcribe_fn, max_workers=3):
    try:
        logging.info("\n\n\n\n------------------------Parallel Transcription Started------------------------")
        
        # Ensure knowledge base directory exists
        os.makedirs(knowledge_base_path, exist_ok=True)
        
        # Search for all audio files & Filter them
        df_files = search_for_audio_files(bata_path)
        df_files['transcribed'] = df_files['file_name'].apply(
            lambda name: os.path.exists(Path(knowledge_base_path) / (Path(name).stem + ".json"))
        )
        df_files_to_process = df_files[~df_files['transcribed']]

        # Finish if all audio files are transcribed
        if df_files_to_process.empty:
            logging.info("All files already transcribed.")
            return

        logging.info("Transcribing %d new files.", len(df_files_to_process))
    
        # Use ThreadPoolExecutor to process files in parallel
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(transcription,
                                row['file_name'],
                                row['file_path'],
                                transcribe_fn,
                                knowledge_base_path): row['file_name']
                for _, row in df_files_to_process.iterrows()
            }
            
            for future in tqdm(as_completed(futures), total=len(futures), desc="Processing audio files", unit='file'):
                future.result()  # This will raise any exceptions encountered during processing
        
        logging.info("All files processed.")
    except Exception as ex:
        logging.error(f"Error in the transcription process: {ex}")

In [13]:
def openai_transcribe(audio_file, file_name):
    with gate:    
        client = OpenAI()
        transcription = client.audio.transcriptions.create(
          model="whisper-1", 
          file=audio_file
        )
    return transcription.text

In [14]:
def elevenlabs_transcribe(audio_file, file_name):
    with gate:                       # blocks if 18 requests already active
        client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))
        result = client.speech_to_text.convert(
            model_id="scribe_v1",
            file=audio_file,
            tag_audio_events=False,
        )
    return result.text 

### Athanasios Mitilinaios

In [10]:
# Configure logging
logging.basicConfig(
    filename="logs/athanasios_mitilinaios_transcription.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

# Paths to audio files and knowledge base where the transcription text will be stored
athanasios_mitilinaios_data = "../data/Omilies/speeches_athanasios_mitilinaios"
athanasios_mitilinaios_knowledge_base = "./knowledge_base/Orthodox/Omilies/speeches_athanasios_mitilinaios"

* ElevenLabs

In [11]:
# parallel_transcription(athanasios_mitilinaios_data,
#                        athanasios_mitilinaios_knowledge_base,
#                        elevenlabs_transcribe)

transcribe_all_sequential(
    data_path=athanasios_mitilinaios_data,
    knowledge_base_path=athanasios_mitilinaios_knowledge_base,
    transcribe_fn=elevenlabs_transcribe,
)

Processing audio files:   0%|                                                     | 1/4183 [01:24<98:11:14, 84.52s/file]


KeyboardInterrupt: 

* OpenAI

In [ ]:
parallel_transcription(athanasios_mitilinaios_data,
                       athanasios_mitilinaios_knowledge_base,
                       openai_transcribe)